<img src="https://www.pucsp.br/sites/default/files/download/brasao-PUCSP-assinatura-principal-RGB.png" style="float:right;" width="150px">

# Atividade: Inteligência Artificial - Machine Learning

**Curso:** Ciência da Computação

**Equipe:** Matheus Heimrath Barbosa e Luan Bonasorte Capella

---

## Classificação Geográfica com Random Forest e Análise Exploratória de Dados (EDA)
Nesta atividade, exploramos o algoritmo **Random Forest** da biblioteca `scikit-learn` utilizando o conjunto de dados **Airports** da biblioteca `vega_datasets`.

O objetivo é prever a localização (estado) de aeroportos baseando-se em suas coordenadas geográficas (latitude e longitude). O projeto está dividido em duas abordagens de modelagem:
1. Avaliação completa com todos os estados americanos presentes no dataset.
2. Avaliação focada nos 5 estados com o maior volume de aeroportos, demonstrando o comportamento do algoritmo diante de fronteiras geográficas isoladas e contíguas.

In [ ]:
# Importando as bibliotecas essenciais
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from vega_datasets import data
import scipy.stats as stats

from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

# Configuração visual dos gráficos
sns.set_theme(style="whitegrid")
plt.rcParams['figure.figsize'] = (10, 6)

### 1. Carregamento e Visão Geral dos Dados

In [ ]:
# Carregando o dataset e removendo nulos de antemão nas colunas chave
df_airports = data.airports().dropna(subset=['latitude', 'longitude', 'state'])

display(df_airports.head())
print("\nInformações gerais do Dataset:")
df_airports.info()

### 2. Análise Exploratória de Dados (EDA)
Antes da modelagem, precisamos entender o comportamento das nossas variáveis. Analisaremos a distribuição das coordenadas, identificaremos outliers e checaremos correlações estatísticas.

#### 2.1. Distribuição de Variáveis e Outliers

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(16, 12))

# Histogramas (Distribuição)
sns.histplot(df_airports['latitude'], kde=True, ax=axes[0, 0], color='skyblue')
axes[0, 0].set_title('Distribuição de Latitude')

sns.histplot(df_airports['longitude'], kde=True, ax=axes[0, 1], color='salmon')
axes[0, 1].set_title('Distribuição de Longitude')

# Boxplots (Detecção de Outliers geográficos)
sns.boxplot(x=df_airports['latitude'], ax=axes[1, 0], color='skyblue')
axes[1, 0].set_title('Boxplot de Latitude (Outliers)')

sns.boxplot(x=df_airports['longitude'], ax=axes[1, 1], color='salmon')
axes[1, 1].set_title('Boxplot de Longitude (Outliers)')

plt.tight_layout()
plt.show()

**Interpretação:** A presença de outliers geográficos é natural neste dataset, pois representam estados muito distantes da massa territorial principal dos Estados Unidos contíguos, como o Alasca (altas latitudes) e o Havaí (baixas longitudes). Não removeremos esses outliers, pois são informações geográficas reais cruciais para a classificação.

#### 2.2. Mapa de Dispersão (Scatterplot)
Plotando latitude contra longitude, recriamos visualmente a malha territorial dos EUA.

In [ ]:
plt.figure(figsize=(12, 7))
sns.scatterplot(data=df_airports, x='longitude', y='latitude', hue='state', legend=False, s=15, alpha=0.7, palette='tab20')
plt.title('Dispersão Geográfica dos Aeroportos (Formação do Mapa)')
plt.xlabel('Longitude')
plt.ylabel('Latitude')
plt.show()

#### 2.3. Matrizes de Correlação (Pearson vs. Spearman)
Verificamos a correlação entre as variáveis de coordenada geométrica.

In [ ]:
coord_df = df_airports[['latitude', 'longitude']]

fig, ax = plt.subplots(1, 2, figsize=(14, 5))

# Pearson (Linearidade)
sns.heatmap(coord_df.corr(method='pearson'), annot=True, cmap='coolwarm', ax=ax[0], vmin=-1, vmax=1)
ax[0].set_title('Matriz de Correlação - Pearson')

# Spearman (Relações Monotônicas)
sns.heatmap(coord_df.corr(method='spearman'), annot=True, cmap='coolwarm', ax=ax[1], vmin=-1, vmax=1)
ax[1].set_title('Matriz de Correlação - Spearman')

plt.show()

---
### 3. Modelagem Parte 1: Predição utilizando TODOS os estados
Neste cenário, desafiamos o modelo a classificar qualquer um dos estados da base, enfrentando o problema natural do desbalanceamento (estados enormes vs pequenos).

In [ ]:
X_all = df_airports[['longitude', 'latitude']]
y_all = df_airports['state']

X_train_all, X_test_all, y_train_all, y_test_all = train_test_split(X_all, y_all, test_size=0.2, random_state=42)

rf_all = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)
rf_all.fit(X_train_all, y_train_all)
y_pred_all = rf_all.predict(X_test_all)

print(f"Acurácia do modelo (Todos os Estados): {accuracy_score(y_test_all, y_pred_all)*100:.2f}%\n")

# Exibindo o report com zero_division=0 para evitar warnings de classes que ficaram sem predições devido ao desbalanceamento
print("Relatório de Classificação (Amostra parcial devido ao alto número de classes):")
print(classification_report(y_test_all, y_pred_all, zero_division=0))

Devido ao grande número de estados, uma matriz de confusão gráfica ficaria sobrecarregada, mas a alta acurácia prova a eficiência espacial do modelo, mesmo com estados contíguos na malha americana.

---
### 4. Modelagem Parte 2: Foco nos 5 Estados com Mais Aeroportos
Para uma visualização e interpretabilidade mais clara das fronteiras de decisão (onde o algoritmo acerta e erra), isolaremos o problema para as 5 maiores classes do dataset.

In [ ]:
top_5_states = df_airports['state'].value_counts().nlargest(5).index
df_top5 = df_airports[df_airports['state'].isin(top_5_states)]

print(f"Os 5 estados com mais aeroportos: {list(top_5_states)}")
print(f"Total de registros reduzidos: {len(df_top5)}")

X_top5 = df_top5[['longitude', 'latitude']]
y_top5 = df_top5['state']

X_train_t5, X_test_t5, y_train_t5, y_test_t5 = train_test_split(X_top5, y_top5, test_size=0.2, random_state=42)

rf_top5 = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)
rf_top5.fit(X_train_t5, y_train_t5)
y_pred_t5 = rf_top5.predict(X_test_t5)

print(f"\nAcurácia do modelo (Top 5 Estados): {accuracy_score(y_test_t5, y_pred_t5)*100:.2f}%")

#### 4.1. Visualização das Fronteiras (Matriz de Confusão)

In [ ]:
plt.figure(figsize=(8, 6))
cm = confusion_matrix(y_test_t5, y_pred_t5, labels=rf_top5.classes_)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', 
            xticklabels=rf_top5.classes_, 
            yticklabels=rf_top5.classes_)
plt.title('Matriz de Confusão - Predição (Top 5 Estados)')
plt.xlabel('Estado Predito')
plt.ylabel('Estado Real')
plt.show()

**Análise de Fronteira:** Note que os erros de classificação ocorrem exclusivamente nas fronteiras compartilhadas. Por exemplo, classificações trocadas entre Texas (TX) e Oklahoma (OK) são consequência dos cortes ortogonais realizados pelas Árvores de Decisão ao longo de divisas estaduais irregulares.

#### 4.2. Feature Importance

In [ ]:
importances = pd.Series(rf_top5.feature_importances_, index=X_top5.columns)
importances.sort_values(ascending=False).plot(kind='bar', color=['skyblue', 'salmon'], figsize=(8,5))
plt.title('Importância das Features (Latitude vs Longitude)')
plt.ylabel('Score Gini')
plt.xticks(rotation=0)
plt.show()

### Conclusão Geral
A Análise Exploratória (EDA) validou a qualidade das coordenadas geográficas e justificou os aparentes outliers como dados reais (Alasca/Havaí). O algoritmo Random Forest provou ser uma escolha excepcional para problemas de fronteira não-linear. Na simulação com todos os estados americanos, o algoritmo obteve altíssimo sucesso ao mapear espacialmente cada coordenada. Ao afunilarmos para os Top 5, ficou ainda mais evidente no relatório e na matriz que as margens de erro residem quase que inteiramente na contiguidade territorial (divisas exatas), validando a robustez do modelo em absorver regras espaciais rígidas.